# Paper-Like Audio-Visual Contact Classification

This notebook follows the paper architecture more closely: AST + CLAP + ViT-B/16 with lightweight Transformer fusion. Original data files are not modified.

In [ ]:
from pathlib import Path
import sys

DATA_ROOT = Path('/kaggle/input/contact-data/dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('../dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('dataset')

CODE_DIR = Path('/kaggle/working/coding')
if not CODE_DIR.exists():
    CODE_DIR = Path('.')
sys.path.append(str(CODE_DIR.resolve()))

print('DATA_ROOT =', DATA_ROOT.resolve())
print('CODE_DIR =', CODE_DIR.resolve())

If Kaggle does not already have `transformers`, install it. Pretrained weights need internet or a Kaggle input/cache containing the model files.

In [ ]:
# Uncomment only if needed on your Kaggle image.
# !pip install -q transformers

In [ ]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from config import DataConfig, TrainConfig, ModelConfig, LABELS
from data import AudioVisualDataset, build_index, make_train_val_split
from metrics import binary_contact_f1, collect_predictions, f1_scores
from paper_model import PaperLikeFusionNet

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

data_cfg = DataConfig(
    data_root=DATA_ROOT,
    target_sample_rate=22000,
    audio_window_sec=0.8,
    train_crop='random',
    eval_crop='center',
    skip_missing_files=True,
)
train_cfg = TrainConfig(epochs=5, batch_size=4, lr=1e-4, num_workers=2, use_amp=True)
model_cfg = ModelConfig(fusion_dim=256, fusion_heads=4, fusion_layers=2, freeze_pretrained=False)
set_seed(train_cfg.seed)

In [ ]:
index = build_index(data_cfg.data_root, skip_missing_files=data_cfg.skip_missing_files)
print('Valid samples:', len(index))
print('Skipped missing files:', index.attrs.get('skipped_missing_files', 0))
display(index['split_name'].value_counts())
display(index['label'].value_counts().reindex(LABELS).fillna(0).astype(int))

In [ ]:
train_df, val_df = make_train_val_split(index, train_cfg.val_size, train_cfg.seed)
train_ds = AudioVisualDataset(train_df, data_cfg, train=True)
val_ds = AudioVisualDataset(val_df, data_cfg, train=False)

train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size, shuffle=True, num_workers=train_cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers, pin_memory=True)

batch = next(iter(train_loader))
print('waveform', batch['waveform'].shape, 'image', batch['image'].shape, 'label', batch['label'].shape)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PaperLikeFusionNet(
    num_classes=len(LABELS),
    sample_rate=data_cfg.target_sample_rate,
    ast_model_name=model_cfg.ast_model_name,
    clap_model_name=model_cfg.clap_model_name,
    fusion_dim=model_cfg.fusion_dim,
    fusion_heads=model_cfg.fusion_heads,
    fusion_layers=model_cfg.fusion_layers,
    fusion_dropout=model_cfg.fusion_dropout,
    freeze_pretrained=model_cfg.freeze_pretrained,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
print('device =', device)

In [ ]:
def run_epoch(loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    scaler = torch.cuda.amp.GradScaler(enabled=train_cfg.use_amp and device == 'cuda')
    total_loss, total_items = 0.0, 0
    logits_list, labels_list = [], []
    for batch in tqdm(loader, leave=False):
        waveform = batch['waveform'].to(device)
        image = batch['image'].to(device)
        labels = batch['label'].to(device)
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=train_cfg.use_amp and device == 'cuda'):
                logits = model(waveform=waveform, image=image)
                loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * labels.size(0)
        total_items += labels.size(0)
        logits_list.append(logits.detach().cpu())
        labels_list.append(labels.detach().cpu())
    y_true, y_pred = collect_predictions(logits_list, labels_list)
    out = f1_scores(y_true, y_pred, num_classes=len(LABELS))
    out['binary_contact_f1'] = binary_contact_f1(y_true, y_pred)
    out['accuracy'] = float((y_true == y_pred).mean())
    out['loss'] = total_loss / total_items
    return out

In [ ]:
out_dir = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
out_dir.mkdir(exist_ok=True, parents=True)
best_macro_f1 = 0.0

for epoch in range(1, train_cfg.epochs + 1):
    train_metrics = run_epoch(train_loader, optimizer)
    val_metrics = run_epoch(val_loader, None)
    print(
        f"epoch={epoch} "
        f"train_loss={train_metrics['loss']:.4f} train_macro_f1={train_metrics['macro_f1']:.4f} "
        f"train_binary_f1={train_metrics['binary_contact_f1']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_macro_f1={val_metrics['macro_f1']:.4f} "
        f"val_weighted_f1={val_metrics['weighted_f1']:.4f} val_binary_f1={val_metrics['binary_contact_f1']:.4f}"
    )
    if val_metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = val_metrics['macro_f1']
        torch.save({'model': model.state_dict(), 'labels': LABELS, 'data_config': data_cfg.__dict__, 'model_config': model_cfg.__dict__, 'val_metrics': val_metrics}, out_dir / 'best_paper_like_model.pt')

print('Best validation macro F1:', best_macro_f1)